<img src= "image1.png">

## 1. Splitting on tokens

In [1]:
from langchain_text_splitters import TokenTextSplitter
import tiktoken


example_string = "Mary had a little lamb, it's fleece was white as snow."



In [2]:
encoding = tiktoken.encoding_for_model('gpt-4o-mini') 
print(encoding)

<Encoding 'o200k_base'>


In [3]:
splitter = TokenTextSplitter(encoding_name= encoding.name,
                             chunk_size = 10,
                             chunk_overlap= 2)

chunks = splitter.split_text(text= example_string)

for i, chunk in enumerate (chunks):
    print(f"Chunk {i+1}: \n{chunk}\n")

Chunk 1: 
Mary had a little lamb, it's fleece was white

Chunk 2: 
 was white as snow.



In [4]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:\nNo. tokens: {len(encoding.encode(chunk))}\n{chunk}\n")

Chunk 1:
No. tokens: 10
Mary had a little lamb, it's fleece was white

Chunk 2:
No. tokens: 5
 was white as snow.



## 2. Semantic Splitting

In [ ]:
# Installer !pip install langchain_experimental 


from langchain_experimental.text_splitter import SemanticChunker

from langchain_ollama import OllamaEmbeddings


# 1. embedding: convert text/ Docs to vector
embeddings = OllamaEmbeddings(
    model= "nomic-embed-text"
)


# 2. semantic splitter
semantic_splitter = SemanticChunker(
    embeddings= embeddings,  # le modèle de vectorisation
    breakpoint_threshold_type= "gradient",
    breakpoint_threshold_amount= 0.8
)


# For a text
chunks = semantic_splitter.split_text(text= example_string)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: \n{chunk}\n")
    
    


C:\Users\hp\AppData\Local\Temp\ipykernel_6700\748128819.py:4: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Chunk 1: 
Mary had a little lamb, it's fleece was white as snow.




<b>1. embeddings:</b>

Pour savoir si deux phrases parlent de la même chose, le SemanticChunker doit d'abord calculer l'embedding de chaque phrase du document. Il compare ensuite <b>la distance vectorielle</b> entre la phrase N et la phrase N+1.

    - Si les deux phrases sont proches dans l'espace vectoriel --> elles restent ensemble.
    
    - Si la distance s'envol "soudainement" --> le thème a changé, il faut couper !


<b>2. breakpoint_threshold_type = "gradient":</b>  

C'est la méthode mathématique utilisée pour détecter une rupture de sujet (le breakpoint).

Il existe principalement 3 types :
    
    - "gradient" : Il mesure la vitesse de changement (la pente/dérivée) entre les phrases. Il repère les moments où la différence s'accélère brusquement. 

    - "percentile"  : Il calcule les distances de tout le document et ne coupe que dans les X% des différences les plus extrêmes.
    
    - "standard_deviation" : Il coupe si la différence entre deux phrases dépasse un certain nombre d'écarts-types par rapport à la moyenne du document.



<b>3. breakpoint_threshold_amount = 0.8:</b>

C'est la sensibilité / le seuil d'exigence de la coupe.   

    - Plus la valeur est élevée (ex: 0.8 ou 0.9), plus il faut un changement de sujet brutal et marqué pour qu'il coupe. On obtiendra ainsi des chunks plus grands et plus rares.

    - Plus la valeur est basse (ex: 0.2 ou 0.3), plus l'algorithme devient sensible à la moindre petite nuance. On obtiendra alors plein de petits chunks.

In [6]:

# for a document


from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("pdf_file_example.pdf")
documents = pdf_loader.load()


chunks = semantic_splitter.split_documents(documents= documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: \n{chunk}\n")

Chunk 1: 
page_content='Le SemanticChunker (découpage sémantique) est l'une des méthodes de chunking les plus 
intelligentes : au lieu de couper le texte aveuglément tous les X caractères ou tokens.' metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-07-22T13:27:46+01:00', 'author': 'RAJAA LEBNAITI', 'moddate': '2026-07-22T13:27:46+01:00', 'source': 'pdf_file_example.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}

Chunk 2: 
page_content='il analyse le 
sens des phrases pour ne couper que là où l'idée change radicalement.' metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-07-22T13:27:46+01:00', 'author': 'RAJAA LEBNAITI', 'moddate': '2026-07-22T13:27:46+01:00', 'source': 'pdf_file_example.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}

